# 03 — Feature Engineering

Applies `build_features()` from `src/features.py` to the assembled trio DataFrame
and analyses the resulting feature matrix.

**Covers:**
1. Load assembled DataFrame from `data/trio_variants.parquet`
2. Call `build_features(df)` → `(X, y)`
3. Feature matrix overview (shape, dtypes, NaN rates)
4. Class imbalance inspection
5. Numeric feature distributions
6. One-hot / ordinal feature summaries
7. HPO overlap score analysis
8. Correlation heatmap
9. Save `X` and `y` as Parquet/CSV for modelling

> **Prerequisite:** run `02_trio_assembly.ipynb` first to produce `data/trio_variants.parquet`.

## 1. Imports & Configuration

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

DATA_DIR    = ROOT / "data"
IN_PARQUET  = DATA_DIR / "trio_variants.parquet"
OUT_X       = DATA_DIR / "features_X.parquet"
OUT_Y       = DATA_DIR / "features_y.csv"
OUT_GROUPS  = DATA_DIR / "features_groups.csv"

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

print("ROOT:", ROOT)

## 2. Load Assembled DataFrame & Build Features

In [ ]:
from src.features import build_features, feature_groups

df = pd.read_parquet(IN_PARQUET)
print(f"Input: {df.shape[0]:,} rows × {df.shape[1]} columns")

X, y = build_features(df)
groups = df["trio_id"]   # preserved for GroupKFold later

print(f"\nFeature matrix X : {X.shape}")
print(f"Target vector  y : {y.shape}  (positive rate = {y.mean():.1%})")
X.head(3)

## 3. Feature Matrix Overview

In [ ]:
fgroups = feature_groups(X)
print("Numeric features  :", fgroups["numeric"])
print("One-hot features  :", fgroups["onehot"])

nan_pct = X.isnull().mean().sort_values(ascending=False) * 100
display(nan_pct[nan_pct > 0].rename("NaN %").to_frame())

print("\nDescriptive statistics (numeric only):")
display(X[fgroups["numeric"]].describe().round(4))

## 4. Class Imbalance

In [ ]:
vc = y.value_counts().sort_index()
labels = ["Other (0)", "Pathogenic (1)"]
colors = ["steelblue", "coral"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(labels, vc.values, color=colors, edgecolor="white")
axes[0].set_title("Class distribution")
axes[0].set_ylabel("Count")
for i, v in enumerate(vc.values):
    axes[0].text(i, v + max(vc)*0.01, str(v), ha="center", fontsize=11)

axes[1].pie(vc.values, labels=labels, colors=colors,
            autopct="%1.1f%%", startangle=90)
axes[1].set_title("Class ratio")

plt.tight_layout()
plt.show()
print(f"\nImbalance ratio (majority/minority): {vc.max()/vc.min():.1f}:1")

## 5. Numeric Feature Distributions by Label

In [ ]:
num_feats = fgroups["numeric"]
n_cols = 4
n_rows = (len(num_feats) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3))
palette = {0: "steelblue", 1: "coral"}

for ax, feat in zip(axes.flat, num_feats):
    for lbl in [0, 1]:
        vals = X.loc[y == lbl, feat].dropna()
        ax.hist(vals, bins=25, alpha=0.55, label=f"y={lbl}", color=palette[lbl])
    ax.set_title(feat, fontsize=9)
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=7)

# hide unused axes
for ax in axes.flat[len(num_feats):]:
    ax.set_visible(False)

plt.suptitle("Numeric feature distributions (y=0 vs y=1)", y=1.01)
plt.tight_layout()
plt.show()

## 6. HPO Overlap Score Analysis

In [ ]:
if "hpo_overlap_score" in X.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    for lbl, color in [(0, "steelblue"), (1, "coral")]:
        vals = X.loc[y == lbl, "hpo_overlap_score"].dropna()
        ax.hist(vals, bins=25, alpha=0.6, label=f"y={lbl}", color=color)
    ax.set_xlabel("HPO overlap score")
    ax.set_title("HPO overlap score distribution by pathogenicity label")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print("\nHPO overlap score stats by label:")
    print(X["hpo_overlap_score"].groupby(y).describe().round(4))
else:
    print("hpo_overlap_score not present in feature matrix")

## 7. Correlation Heatmap (Numeric Features)

In [ ]:
corr_df = X[fgroups["numeric"]].assign(y=y.values).corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.zeros_like(corr_df, dtype=bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr_df, mask=mask, annot=True, fmt=".2f",
            cmap="coolwarm", center=0, linewidths=0.4,
            annot_kws={"size": 7}, ax=ax)
ax.set_title("Lower-triangle correlation matrix (numeric features + label)")
plt.tight_layout()
plt.show()

## 8. Save Features for Modelling

In [ ]:
X.to_parquet(OUT_X, index=False)
y.to_csv(OUT_Y, index=False, header=True)
groups.to_csv(OUT_GROUPS, index=False, header=True)

print(f"X      → {OUT_X}  {X.shape}")
print(f"y      → {OUT_Y}  {y.shape}")
print(f"groups → {OUT_GROUPS}  {groups.shape}")